# Inference Bioinspired3D - Finetuned Model for Blender Code Generation

Rachel K. Luu, 2026

In [ ]:
BASE_MODEL = "meta-llama/Llama-3.2-3B-Instruct"
LORA_ADAPTER = "rachelkluu/bioinspired3D"

JSONL_PATHS = [
    "./RAG/biodataset_RAG.jsonl",
]
TOP_K = 2
EMBED_MODEL_NAME = "BAAI/bge-small-en-v1.5"

## BUILD RAG DATABASE FOR Bioinspired3D

def load_jsonl(path):
    """Load a single JSONL file."""
    data = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            try:
                entry = json.loads(line)
                entry["source"] = os.path.basename(path)
                data.append(entry)
            except json.JSONDecodeError:
                print(f"⚠️ Skipping invalid line in {path}")
    return data

def load_all_jsonls(paths):
    """Combine multiple JSONL files."""
    combined = []
    for p in paths:
        if os.path.exists(p):
            print(f"📂 Loading: {p}")
            combined.extend(load_jsonl(p))
        else:
            print(f"⚠️ File not found: {p}")
    print(f"✅ Loaded {len(combined)} total entries from {len(paths)} files.")
    return combined

base_codes = load_all_jsonls(JSONL_PATHS)

print("🔍 Encoding entries...")
embed_model = SentenceTransformer(EMBED_MODEL_NAME)

def get_embedding(item):
    """Generate embedding from instruction + code."""
    text = f"{item.get('instruction', '')}\n{item.get('code', '')}"
    return embed_model.encode(text, normalize_embeddings=True)

embeddings = np.vstack([get_embedding(item) for item in base_codes]).astype("float32")
index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)
print(f"✅ Built FAISS index with {len(base_codes)} items.")

def retrieve(query, k=TOP_K):
    """Retrieve top-k similar entries."""
    q_emb = embed_model.encode([query], normalize_embeddings=True).astype("float32")
    D, I = index.search(q_emb, k)
    results = []
    for rank, idx in enumerate(I[0]):
        item = base_codes[idx]
        results.append({
            "rank": rank,
            "score": float(D[0][rank]),
            "source": item["source"],
            "category": item.get("category", "unknown"),
            "instruction": item.get("instruction", ""),
            "code": item.get("code", "")
        })
    return results

def build_context(query, k=TOP_K):
    """Build formatted RAG context for LLM input."""
    retrieved = retrieve(query, k=k)
    context = []
    for r in retrieved:
        context.append(
            f"Source: {r['source']}\n"
            f"Instruction: {r['instruction']}\n"
            f"Category: {r['category']}\n"
            f"Code:\n{r['code']}\n"
        )
    return "\n---\n".join(context)


from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

bio3d_tok = AutoTokenizer.from_pretrained(BASE_MODEL)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,       # full precision (fp16)
    device_map={"": DEVICE_3D},      # which GPU to use
)

# Load the LoRA adapter on top of the base model
from peft import PeftModel
bio3d_model = PeftModel.from_pretrained(base_model, LORA_ADAPTER)

bio3d_model.eval()

print(f"✅ Loaded Bio3D LoRA (3B full precision fp16)")
